# Errors and Debugging

The most common type of errors at runtime stems from worker tasks.
A first measure to preventing this is using the correct types, but still something might go wrong.
In this example we will produce an error and investigate it with the debugging tools in tierkreis.

## Worker Errors

Worker errors can occur in multiple ways.
For python workers an error occurs when an uncaught exception raises.
For other workers (including python) a non-zero exit code will also produce an error.

Defining a graph that will always run an error:

In [ ]:
from example_workers.error_worker.api.stubs import fail

from tierkreis.builder import GraphBuilder
from tierkreis.controller.data.core import EmptyModel
from tierkreis.controller.data.models import TKR


def error_graph() -> GraphBuilder:
    g = GraphBuilder(EmptyModel, TKR[str])
    output = g.task(fail())
    g.outputs(output)
    return g

The task `fail` will raise an `TierkreisError` (`"I refuse!"`) when running:

In [ ]:
from pathlib import Path
from uuid import UUID

from tierkreis.controller import run_graph
from tierkreis.controller.executor.uv_executor import UvExecutor
from tierkreis.controller.storage.filestorage import ControllerFileStorage
from tierkreis.exceptions import TierkreisError

workflow_id = UUID(int=103)
storage = ControllerFileStorage(workflow_id, name="error_handling", do_cleanup=True)

registry_path = Path().parent / "example_workers"
executor = UvExecutor(registry_path=registry_path, logs_path=storage.logs_path)
try:
    run_graph(
        storage,
        executor,
        error_graph().data,
        {"value": "world!"},
        polling_interval_seconds=0.1,
    )
except TierkreisError:  # we will catch this here
    output = storage.read_errors()

## Debugging

In this example we will only investigate the root cause of the error.
In the next one we will see how we can resume a graph from its checkpoint.

The first avenue for debugging is enabling fine grained logging.
The tierkreis logging inherits properties from the root logger so it suffices to set a `basicConfig` which changes **only** the logger of the controller.
When running a python worker, Tierkreis will check the environment variables `$TKR_LOG_LEVEL`, `$TKR_LOG_FORMAT` and `$TKR_DATE_FORMAT` for logger information as detailed [here](../logging_and_errors.md).

In [ ]:
import contextlib
import logging

logging.basicConfig(
    format="%(asctime)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S%z",
    level=logging.DEBUG,
)

storage.clean_graph_files()
with contextlib.suppress(TierkreisError):
    run_graph(
        storage,
        executor,
        error_graph(),
        {"value": "world!"},
        polling_interval_seconds=0.1,
    )

For most use cases, tierkreis can also leverage python breakpoint debugging.
The condition for this to work is that the graph only uses python workers.
To do this you can use an alternative executor that stores the graph information in memory

In [ ]:
from tierkreis.controller.executor.in_memory_executor import InMemoryExecutor
from tierkreis.storage import InMemoryStorage

storage = InMemoryStorage(UUID(int=103))
executor = InMemoryExecutor(registry_path, storage)

try:
    run_graph(
        storage,
        executor,
        error_graph().data,
        {"value": "world!"},
    )
except Exception:  # Note the different exception type here
    pass